# Cross-domain breast ultrasound experiments

This notebook implements three experimental approaches on BUSI (and BreakHis) datasets:

1. **SSDAVT** — Self-Supervised Domain-Adaptive ViT (cross-domain BreakHis → BUSI)  
2. **MSVTE-U** — Multi-Scale ViT Ensemble with Uncertainty (BUSI-only)  
3. **HCML** — Hybrid Contrastive Meta-Learning (SimCLR + ProtoNet few-shot on BUSI)

All approaches:

- Log training and validation metrics  
- Save best model checkpoints  
- Save plots to `RESULTS_DIR`  
- Can be safely resumed without re-running previous sections (if checkpoints exist)

## Names used here vs. names used in the paper

| In this notebook | In the paper |
|---|---|
| `MSVTE-U` (BUSI-only) | HViTE-U -- the baseline arm |
| `CD-MSVTE-U` (cross-domain) | HViTE-U + DA -- the adversarial arm |
| `SSDAVT` | SSDAVT |
| `CD-HCML` SimCLR stage | Contrastive pretraining |
| `CD-HCML` ProtoNet stage | **Not reported.** See the README: this routine has a
  class-mapping defect and its output is not used anywhere in the paper. |


In [1]:
import os
import random
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from torchvision import transforms as T
from PIL import Image

import timm
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# ---------------------------
# Device & Reproducibility
# ---------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ---------------------------
# Global Config
# ---------------------------
from pathlib import Path

class CFG:
    # ==== PATHS: change these two to your local dataset locations ====
    PROJECT_ROOT = Path(r"D:\My Thesis\hybrid-three-approach")  # where checkpoints and results are written

    DATA_ROOT    = Path(r"D:\My Thesis\bushra\data")

    # BreaKHis_v1, down to the folder containing benign/ and malignant/
    BREAKHIS_ROOT = DATA_ROOT / "breakhis" / "BreaKHis_v1" / "histology_slides" / "breast"

    # BUSI, the Original/ directory; the ground-truth masks are not used
    BUSI_ROOT     = DATA_ROOT / "busi" / "Dataset_BUSI_with_GT" / "Original"

    image_size = 224

    # ==== HYPERPARAMETERS ====
    ssdavt_model_name = "vit_small_patch16_224.dino"
    ssdavt_batch_size = 16
    ssdavt_epochs = 50
    ssdavt_lr = 3e-4
    ssdavt_weight_decay = 1e-4
    ssdavt_lambda_domain = 0.3   # lambda_max for the gradient reversal layer

    msvte_batch_size = 16
    msvte_epochs = 30
    msvte_lr = 3e-4
    msvte_weight_decay = 1e-4
    msvte_mc_passes = 10

    simclr_epochs = 20
    simclr_batch_size = 64
    simclr_lr = 3e-4
    simclr_weight_decay = 1e-4
    simclr_temperature = 0.5

    meta_n_way = 3
    meta_k_shot = 5
    meta_q_queries = 10
    meta_iters = 400
    meta_batch_episodes = 8
    meta_lr_backbone = 1e-4
    meta_lr_head = 5e-4
    meta_weight_decay = 1e-4


# Directories
CHECKPOINT_DIR = CFG.PROJECT_ROOT / "models"
RESULTS_DIR    = CFG.PROJECT_ROOT / "results"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("PROJECT_ROOT   :", CFG.PROJECT_ROOT)
print("CHECKPOINT_DIR :", CHECKPOINT_DIR)
print("RESULTS_DIR    :", RESULTS_DIR)


c:\ProgramData\anaconda3\envs\fewshot-meta\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Using device: cuda
PROJECT_ROOT   : D:\My Thesis\hybrid-three-approach
CHECKPOINT_DIR : D:\My Thesis\hybrid-three-approach\models
RESULTS_DIR    : D:\My Thesis\hybrid-three-approach\results


In [2]:
BUSI_CLASSES = ["benign", "malignant", "normal"]
BUSI_CLASS_TO_IDX = {c: i for i, c in enumerate(BUSI_CLASSES)}

def list_busi_samples(busi_root: Path):
    samples = []
    for cls_name in BUSI_CLASSES:
        cls_dir = busi_root / cls_name
        if not cls_dir.exists():
            continue
        for fname in os.listdir(cls_dir):
            if not fname.lower().endswith(".png"):
                continue
            if fname.endswith("_mask.png"):
                continue
            samples.append((str(cls_dir / fname), BUSI_CLASS_TO_IDX[cls_name]))
    return samples

busi_all = list_busi_samples(CFG.BUSI_ROOT)
print(f"Total BUSI images: {len(busi_all)}")

paths = [p for (p, y) in busi_all]
labels = [y for (p, y) in busi_all]

train_paths, test_paths, train_labels, test_labels = train_test_split(
    paths, labels, test_size=0.2, random_state=SEED, stratify=labels
)
train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_paths, train_labels, test_size=0.25, random_state=SEED, stratify=train_labels
)

busi_train_s = list(zip(train_paths, train_labels))
busi_val_s   = list(zip(val_paths, val_labels))
busi_test_s  = list(zip(test_paths, test_labels))

print(f"BUSI train: {len(busi_train_s)}, val: {len(busi_val_s)}, test: {len(busi_test_s)}")


Total BUSI images: 798
BUSI train: 478, val: 160, test: 160


In [3]:
class BUSIDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


# ============================================================
#  BreakHisDataset -- reads the BreaKHis_v1 SOB folder layout
# ============================================================
class BreakHisDataset(Dataset):
    """
    BreakHis dataloader adapted to this real structure:

    breakhis/BreaKHis_v1/histology_slides/breast/
        benign/SOB/adenosis/SOB_B_*/40X/*.png
        malignant/SOB/ductal_carcinoma/SOB_M_*/40X/*.png
    """

    def __init__(self, root: Path, transform=None):
        self.root = Path(root)
        self.transform = transform
        self.samples = []
        self.classes = []
        self.class_to_idx = {}

        # class names under benign + malignant
        self.benign_classes = ["adenosis", "fibroadenoma", "phyllodes_tumor", "tubular_adenoma"]
        self.malignant_classes = ["ductal_carcinoma", "lobular_carcinoma",
                                  "mucinous_carcinoma", "papillary_carcinoma"]

        self._load_data()

    def _register_class(self, class_name: str) -> int:
        """Assigns index to a class if not already assigned."""
        if class_name not in self.class_to_idx:
            idx = len(self.classes)
            self.class_to_idx[class_name] = idx
            self.classes.append(class_name)
        return self.class_to_idx[class_name]

    def _load_data(self):
        benign_root = self.root / "benign" / "SOB"
        malignant_root = self.root / "malignant" / "SOB"

        print(f"[BreakHis] Scanning benign folder:    {benign_root}")
        print(f"[BreakHis] Scanning malignant folder: {malignant_root}")

        self._load_group(benign_root, self.benign_classes)
        self._load_group(malignant_root, self.malignant_classes)

        print(f"[BreakHis] Loaded {len(self.samples)} images from {len(self.classes)} classes.")
        print("[BreakHis] Classes:", self.classes)

    def _load_group(self, group_root: Path, class_list):
        """Loads images from benign or malignant folder."""
        if not group_root.exists():
            print(f"[BreakHis] WARNING: Folder not found: {group_root}")
            return

        # Loop through each class folder inside SOB/
        for class_name in class_list:
            class_dir = group_root / class_name
            if not class_dir.exists():
                print(f"[BreakHis] WARNING: Missing class folder: {class_dir}")
                continue

            class_idx = self._register_class(class_name)

            # Example:
            # adenosis/
            #   └── SOB_B_A_14-22549AB/
            #        └── 40X/*.png
            sob_dirs = list(class_dir.glob("SOB_*"))
            if not sob_dirs:
                print(f"[BreakHis] WARNING: No SOB_* folders under {class_dir}")

            for sob_dir in sob_dirs:
                # magnification folders like 40X, 100X...
                for mag_dir in sob_dir.iterdir():
                    if not mag_dir.is_dir():
                        continue
                    # load all PNG images
                    for img_path in mag_dir.glob("*.png"):
                        self.samples.append((str(img_path), class_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


In [4]:
def get_basic_tf(is_train=True):
    if is_train:
        return T.Compose([
            T.Resize((CFG.image_size, CFG.image_size)),
            T.RandomHorizontalFlip(),
            T.RandomRotation(15),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225]),
        ])
    else:
        return T.Compose([
            T.Resize((CFG.image_size, CFG.image_size)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225]),
        ])

def get_simclr_tf():
    return T.Compose([
        T.RandomResizedCrop(CFG.image_size, scale=(0.5, 1.0)),
        T.RandomHorizontalFlip(),
        T.RandomApply([T.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
        T.RandomGrayscale(p=0.2),
        T.GaussianBlur(kernel_size=9),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]),
    ])


In [5]:
class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None

def grad_reverse(x, lambd=1.0):
    return GradReverse.apply(x, lambd)


class SSDAVTModel(nn.Module):
    def __init__(self, backbone_name, num_classes=3):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0)
        feat_dim = self.backbone.num_features
        self.class_head = nn.Linear(feat_dim, num_classes)
        self.domain_head = nn.Linear(feat_dim, 2)  # 0: BreakHis, 1: BUSI

    def forward_features(self, x):
        return self.backbone(x)

    def forward_class(self, x):
        f = self.forward_features(x)
        return self.class_head(f)

    def forward_domain(self, x, lambd=1.0):
        f = self.forward_features(x)
        f_rev = grad_reverse(f, lambd)
        return self.domain_head(f_rev)


In [6]:
def get_domain_tf():
    return get_basic_tf(is_train=True)


def compute_busi_class_weights(samples):
    labels = [lbl for (_, lbl) in samples]
    counts = Counter(labels)
    total = sum(counts.values())
    weights = []
    for cls_idx in range(len(BUSI_CLASSES)):
        cls_count = counts.get(cls_idx, 1)
        weights.append(total / cls_count)
    w = torch.tensor(weights, dtype=torch.float32, device=device)
    print("BUSI train class counts:", counts)
    print("Class weights (inverse freq):", w)
    return w


In [7]:
def train_ssdavt():
    print("\n=== Approach 1: SSDAVT Training ===")

    # ---------------------------
    # Load BUSI datasets
    # ---------------------------
    busi_train_ds = BUSIDataset(busi_train_s, transform=get_domain_tf())
    busi_val_ds   = BUSIDataset(busi_val_s,   transform=get_domain_tf())
    busi_test_ds  = BUSIDataset(busi_test_s,  transform=get_basic_tf(is_train=False))

    # ---------------------------
    # Load BreakHis dataset
    # ---------------------------
    breakhis_ds = BreakHisDataset(CFG.BREAKHIS_ROOT, transform=get_domain_tf())
    print(f"[DEBUG] BreakHis samples found: {len(breakhis_ds)}")

    if len(breakhis_ds) == 0:
        raise RuntimeError(
            f"❌ ERROR: BreakHis dataset is EMPTY.\n"
            f"Checked path: {CFG.BREAKHIS_ROOT}\n"
            f"Fix your dataset folder before training."
        )

    # ---------------------------
    # DataLoaders
    # ---------------------------
    busi_train_loader = DataLoader(
        busi_train_ds, batch_size=CFG.ssdavt_batch_size,
        shuffle=True, num_workers=0, pin_memory=True
    )
    busi_val_loader = DataLoader(
        busi_val_ds, batch_size=CFG.ssdavt_batch_size,
        shuffle=False, num_workers=0, pin_memory=True
    )
    busi_test_loader = DataLoader(
        busi_test_ds, batch_size=CFG.ssdavt_batch_size,
        shuffle=False, num_workers=0, pin_memory=True
    )
    breakhis_loader = DataLoader(
        breakhis_ds, batch_size=CFG.ssdavt_batch_size,
        shuffle=True, num_workers=0, pin_memory=True
    )

    # ---------------------------
    # Model
    # ---------------------------
    model = SSDAVTModel(CFG.ssdavt_model_name, num_classes=len(BUSI_CLASSES)).to(device)

    # ---------------------------
    # Class weights (fix imbalance)
    # ---------------------------
    class_weights = compute_busi_class_weights(busi_train_s)
    cls_criterion = nn.CrossEntropyLoss(weight=class_weights)
    dom_criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=CFG.ssdavt_lr, weight_decay=CFG.ssdavt_weight_decay
    )

    history = {"train_cls_loss": [], "train_dom_loss": [], "val_acc": []}
    best_val_acc = 0.0
    best_path = CHECKPOINT_DIR / "ssdavt_best.pth"

    breakhis_iter = iter(breakhis_loader)

    # ---------------------------
    # Epoch Loop
    # ---------------------------
    for epoch in range(1, CFG.ssdavt_epochs + 1):
        model.train()
        running_cls, running_dom, total = 0.0, 0.0, 0

        pbar = tqdm(busi_train_loader, desc=f"[SSDAVT] Epoch {epoch}/{CFG.ssdavt_epochs}")

        for imgs_busi, labels_busi in pbar:

            imgs_busi = imgs_busi.to(device)
            labels_busi = labels_busi.to(device)
            bs = imgs_busi.size(0)

            # ---------------------------
            # Fetch BreakHis Batch (safe)
            # ---------------------------
            try:
                imgs_bh, _ = next(breakhis_iter)
            except StopIteration:
                breakhis_iter = iter(breakhis_loader)
                imgs_bh, _ = next(breakhis_iter)

            imgs_bh = imgs_bh.to(device)
            bh_size = imgs_bh.size(0)

            # ---------------------------
            # 1️⃣ Classification Loss (BUSI only)
            # ---------------------------
            cls_logits = model.forward_class(imgs_busi)
            cls_loss = cls_criterion(cls_logits, labels_busi)

            # ---------------------------
            # 2️⃣ Domain Adversarial Loss (BUSI + BreakHis)
            # ---------------------------
            dom_imgs = torch.cat([imgs_bh, imgs_busi], dim=0)
            dom_labels = torch.cat([
                torch.zeros(bh_size, dtype=torch.long, device=device),
                torch.ones(bs, dtype=torch.long, device=device)
            ], dim=0)

            # Gradual schedule: start small, ramp up to CFG.ssdavt_lambda_domain
            progress = epoch / CFG.ssdavt_epochs
            lambd = CFG.ssdavt_lambda_domain * progress  # e.g., goes 0.0 → λ over epochs

            dom_logits = model.forward_domain(dom_imgs, lambd=lambd)
            dom_loss = dom_criterion(dom_logits, dom_labels)

            loss = cls_loss + lambd * dom_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # accumulate stats
            running_cls += cls_loss.item() * bs
            running_dom += dom_loss.item() * (bh_size + bs)
            total += bs

            # update tqdm
            pbar.set_postfix({
                "cls_loss": f"{cls_loss.item():.3f}",
                "dom_loss": f"{dom_loss.item():.3f}"
            })

        # aggregate losses
        avg_cls = running_cls / total
        avg_dom = running_dom / (len(breakhis_ds) + total)
        history["train_cls_loss"].append(avg_cls)
        history["train_dom_loss"].append(avg_dom)

        # ---------------------------
        # Validation
        # ---------------------------
        model.eval()
        correct, total_val = 0, 0
        with torch.no_grad():
            for imgs, labels in busi_val_loader:
                imgs = imgs.to(device)
                labels = labels.to(device)
                logits = model.forward_class(imgs)
                preds = logits.argmax(dim=1)
                correct += (preds == labels).sum().item()
                total_val += labels.size(0)

        val_acc = correct / total_val
        history["val_acc"].append(val_acc)

        print(f"[SSDAVT] Epoch {epoch}/{CFG.ssdavt_epochs} - "
              f"cls={avg_cls:.4f}, dom={avg_dom:.4f}, val_acc={val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_path)
            print("   ✔ Saved BEST SSDAVT model.")

    # ---------------------------
    # Plot training curves
    # ---------------------------
    epochs_axis = range(1, CFG.ssdavt_epochs + 1)

    plt.figure()
    plt.plot(epochs_axis, history["train_cls_loss"], label="Classification Loss")
    plt.plot(epochs_axis, history["train_dom_loss"], label="Domain Loss")
    plt.legend()
    plt.title("SSDAVT Training Losses")
    plt.savefig(RESULTS_DIR / "ssdavt_losses.png")
    plt.close()

    plt.figure()
    plt.plot(epochs_axis, history["val_acc"], label="Val Accuracy")
    plt.legend()
    plt.title("SSDAVT Validation Accuracy")
    plt.savefig(RESULTS_DIR / "ssdavt_val_acc.png")
    plt.close()

    # ---------------------------
    # Testing
    # ---------------------------
    model.load_state_dict(torch.load(best_path, map_location=device))
    model.eval()

    all_labels, all_preds = [], []
    with torch.no_grad():
        for imgs, labels in busi_test_loader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            logits = model.forward_class(imgs)
            preds = logits.argmax(dim=1)

            all_labels += labels.cpu().tolist()
            all_preds += preds.cpu().tolist()

    print("\n=== SSDAVT TEST RESULTS ===")
    print(classification_report(all_labels, all_preds, target_names=BUSI_CLASSES, digits=4))
    cm = confusion_matrix(all_labels, all_preds)
    print(cm)

    # save report
    with open(RESULTS_DIR / "ssdavt_test_report.txt", "w") as f:
        f.write(classification_report(all_labels, all_preds, target_names=BUSI_CLASSES, digits=4))
        f.write("\nConfusion Matrix:\n")
        f.write(str(cm))

    return model


In [8]:
class SingleViTClassifier(nn.Module):
    def __init__(self, backbone_name, num_classes=3, dropout_p=0.3):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0)
        feat_dim = self.backbone.num_features
        self.dropout = nn.Dropout(p=dropout_p)
        self.head = nn.Linear(feat_dim, num_classes)

    def forward(self, x):
        f = self.backbone(x)
        f = self.dropout(f)
        return self.head(f)


def build_msvte_models():
    names = [
        "vit_small_patch16_224.augreg_in21k",
        "vit_base_patch16_224.augreg_in21k",
        "deit_small_patch16_224"
    ]
    models = []
    for n in names:
        m = SingleViTClassifier(n, num_classes=len(BUSI_CLASSES)).to(device)
        models.append(m)
    return models, names


In [9]:
def train_single_vit(model, train_loader, val_loader, class_weights, name, epochs, lr, weight_decay):
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_acc = 0.0
    best_path = CHECKPOINT_DIR / f"msvte_{name}_best.pth"

    history = {"train_loss": [], "val_acc": []}

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss, total, correct = 0.0, 0, 0

        pbar = tqdm(train_loader, desc=f"[{name}] Epoch {epoch}/{epochs}")
        for imgs, labels in pbar:
            imgs = imgs.to(device)
            labels = labels.to(device)

            logits = model(imgs)
            loss = criterion(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            total += labels.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()

            pbar.set_postfix({"loss": f"{loss.item():.3f}",
                              "train_acc": f"{correct/max(1,total):.3f}"})

        avg_loss = running_loss / max(1, total)
        history["train_loss"].append(avg_loss)

        model.eval()
        correct_val, total_val = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs = imgs.to(device)
                labels = labels.to(device)
                logits = model(imgs)
                preds = logits.argmax(dim=1)
                correct_val += (preds == labels).sum().item()
                total_val += labels.size(0)
        val_acc = correct_val / max(1, total_val)
        history["val_acc"].append(val_acc)

        print(f"[{name}] Epoch {epoch}/{epochs} - train_loss={avg_loss:.4f}, val_acc={val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_path)
            print(f"   ↳ New best {name} model saved.")

    epochs_axis = range(1, len(history["train_loss"]) + 1)
    plt.figure()
    plt.plot(epochs_axis, history["train_loss"])
    plt.xlabel("Epoch"); plt.ylabel("Train Loss")
    plt.title(f"{name} Train Loss")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"{name}_train_loss.png")
    plt.close()

    plt.figure()
    plt.plot(epochs_axis, history["val_acc"])
    plt.xlabel("Epoch"); plt.ylabel("Val Accuracy")
    plt.title(f"{name} Val Accuracy")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"{name}_val_acc.png")
    plt.close()

    return best_path


In [10]:
# ============================================================
# Approach 2: Cross-Domain MSVTE-U  (called HViTE-U + DA in the paper)
# Domain-Adaptive Multi-Scale ViT Ensemble with Uncertainty
# ============================================================

# ---- Safe defaults if CFG doesn't define these ----
MSVTE_DROPOUT          = getattr(CFG, "msvte_dropout", 0.3)
MSVTE_LAMBDA_DOMAIN    = getattr(CFG, "msvte_lambda_domain", getattr(CFG, "ssdavt_lambda_domain", 0.3))
MSVTE_DOMAIN_LOSS_WT   = getattr(CFG, "msvte_domain_loss_weight", 1.0)
MSVTE_MC_SAMPLES       = getattr(CFG, "msvte_mc_samples", getattr(CFG, "msvte_mc_passes", 10))
NUM_WORKERS            = getattr(CFG, "num_workers", 0)
GRAD_CLIP              = getattr(CFG, "grad_clip", 1.0)


class DomainAdaptiveViTClassifier(nn.Module):
    """
    Multi-scale ViT/DeiT model with:
      - BUSI classification head (3 classes)
      - Domain head (BreakHis vs BUSI) with Gradient Reversal
      - Dropout for MC-dropout uncertainty
    """
    def __init__(self, backbone_name, num_classes=3, dropout_p=0.3):
        super().__init__()
        self.backbone_name = backbone_name
        self.backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0)
        feat_dim = self.backbone.num_features

        self.dropout = nn.Dropout(p=dropout_p)
        self.class_head = nn.Linear(feat_dim, num_classes)
        self.domain_head = nn.Linear(feat_dim, 2)  # 0=BreakHis, 1=BUSI

    def forward_features(self, x):
        return self.backbone(x)

    def forward_class(self, x):
        f = self.forward_features(x)
        f = self.dropout(f)
        return self.class_head(f)

    def forward_domain(self, x, lambd=1.0):
        f = self.forward_features(x)
        f = grad_reverse(f, lambd=lambd)
        f = self.dropout(f)
        return self.domain_head(f)


def build_cd_msvte_models():
    """
    Same idea as MSVTE-U, but each backbone is trained with domain adaptation.
    """
    names = [
        "vit_small_patch16_224.augreg_in21k",
        "vit_base_patch16_224.augreg_in21k",
        "deit_small_patch16_224"
    ]
    models = [DomainAdaptiveViTClassifier(n, num_classes=len(BUSI_CLASSES), dropout_p=MSVTE_DROPOUT).to(device)
              for n in names]
    return models, names


@torch.no_grad()
def eval_busi_classifier(model, loader):
    model.eval()
    total, correct = 0, 0
    all_labels, all_preds = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        labels = labels.to(device)
        logits = model.forward_class(imgs)
        preds = logits.argmax(dim=1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()
        all_labels.extend(labels.detach().cpu().tolist())
        all_preds.extend(preds.detach().cpu().tolist())
    acc = correct / max(total, 1)
    return acc, all_labels, all_preds


def train_single_cd_vit(model, name,
                        busi_train_loader, busi_val_loader,
                        breakhis_loader,
                        class_weights,
                        epochs, lr, weight_decay,
                        lambda_domain=1.0, domain_loss_weight=1.0,
                        grad_clip=1.0):
    """
    Trains ONE domain-adaptive ViT model:
      - Classification loss on BUSI batches only
      - Domain loss on concatenated BreakHis + BUSI batches (GRL)
    """
    cls_criterion = nn.CrossEntropyLoss(weight=class_weights)
    dom_criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_acc = -1.0
    best_path = CHECKPOINT_DIR / f"cd_msvte_{name.replace('/', '_')}_best.pth"

    breakhis_iter = iter(breakhis_loader)

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        running_cls = 0.0
        running_dom = 0.0
        n_steps = 0

        # Ramp-up GRL lambda (0 -> lambda_domain)
        progress = (epoch - 1) / max(epochs - 1, 1)
        lambd = progress * lambda_domain

        for busi_imgs, busi_labels in busi_train_loader:
            busi_imgs = busi_imgs.to(device)
            busi_labels = busi_labels.to(device)

            try:
                bh_imgs, _ = next(breakhis_iter)
            except StopIteration:
                breakhis_iter = iter(breakhis_loader)
                bh_imgs, _ = next(breakhis_iter)
            bh_imgs = bh_imgs.to(device)

            # (1) BUSI classification loss
            cls_logits = model.forward_class(busi_imgs)
            cls_loss = cls_criterion(cls_logits, busi_labels)

            # (2) Domain loss
            dom_imgs = torch.cat([bh_imgs, busi_imgs], dim=0)
            dom_labels = torch.cat([
                torch.zeros(bh_imgs.size(0), dtype=torch.long, device=device),
                torch.ones(busi_imgs.size(0), dtype=torch.long, device=device)
            ], dim=0)

            dom_logits = model.forward_domain(dom_imgs, lambd=lambd)
            dom_loss = dom_criterion(dom_logits, dom_labels)

            loss = cls_loss + domain_loss_weight * dom_loss

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            if grad_clip is not None and grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

            running_loss += loss.item()
            running_cls += cls_loss.item()
            running_dom += dom_loss.item()
            n_steps += 1

        val_acc, _, _ = eval_busi_classifier(model, busi_val_loader)

        print(f"[CD-MSVTE] {name} | Epoch {epoch:02d}/{epochs} | "
              f"Loss {running_loss/max(n_steps,1):.4f} (cls {running_cls/max(n_steps,1):.4f}, dom {running_dom/max(n_steps,1):.4f}, λ {lambd:.2f}) | "
              f"Val Acc {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_path)

    model.load_state_dict(torch.load(best_path, map_location=device))
    return best_path


def cd_msvte_mc_ensemble_predict(models, loader, mc_samples):
    # Keep dropout ON
    for model in models:
        model.train()

    all_labels, all_preds, all_uncert = [], [], []

    # --- Critical: no grad to avoid graph + reduce memory ---
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device, non_blocking=True)

            probs_mc = []

            for _ in range(mc_samples):
                probs_per_model = []

                for model in models:
                    logits = model.forward_class(imgs)
                    probs = torch.softmax(logits, dim=1)
                    probs_per_model.append(probs)

                # average across models for this MC sample
                probs_mc.append(torch.stack(probs_per_model, dim=0).mean(dim=0))

                # free temporary tensors early (helps fragmentation)
                del probs_per_model, logits, probs

            # shape: [mc_samples, B, C]
            probs_mc = torch.stack(probs_mc, dim=0)

            mean_probs = probs_mc.mean(dim=0)                 # [B, C]
            uncertainty = probs_mc.var(dim=0).mean(dim=1)     # [B]  (mean variance across classes)
            preds = mean_probs.argmax(dim=1)

            all_labels.extend(labels.cpu().numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())
            all_uncert.extend(uncertainty.cpu().numpy().tolist())

            # cleanup per-batch
            del imgs, probs_mc, mean_probs, uncertainty, preds
            torch.cuda.empty_cache()

    return all_labels, all_preds, all_uncert



def train_cd_msvte_u():
    """
    UPDATED Approach 2:
    - Train 3 domain-adaptive ViT/DeiT models with GRL (BreakHis vs BUSI)
    - Ensemble + MC-dropout uncertainty on BUSI test
    """
    print("\n=== Approach 2 (UPDATED): Cross-Domain MSVTE-U Training ===")

    train_tf = get_basic_tf(is_train=True)
    test_tf  = get_basic_tf(is_train=False)

    busi_train_ds = BUSIDataset(busi_train_s, transform=train_tf)
    busi_val_ds   = BUSIDataset(busi_val_s, transform=test_tf)
    busi_test_ds  = BUSIDataset(busi_test_s, transform=test_tf)

    breakhis_ds = BreakHisDataset(CFG.BREAKHIS_ROOT, transform=get_domain_tf())

    class_weights = compute_busi_class_weights(busi_train_s)

    sample_weights = [1.0 / (class_weights[int(lbl)].item() + 1e-8) for _, lbl in busi_train_s]
    sampler = torch.utils.data.WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

    busi_train_loader = DataLoader(busi_train_ds, batch_size=CFG.msvte_batch_size, sampler=sampler,
                                   num_workers=NUM_WORKERS, pin_memory=True)
    busi_val_loader   = DataLoader(busi_val_ds, batch_size=CFG.msvte_batch_size, shuffle=False,
                                   num_workers=NUM_WORKERS, pin_memory=True)
    busi_test_loader  = DataLoader(busi_test_ds, batch_size=CFG.msvte_batch_size, shuffle=False,
                                   num_workers=NUM_WORKERS, pin_memory=True)

    breakhis_loader = DataLoader(breakhis_ds, batch_size=CFG.msvte_batch_size, shuffle=True,
                                 num_workers=NUM_WORKERS, pin_memory=True)

    models, names = build_cd_msvte_models()
    for model, name in zip(models, names):
        train_single_cd_vit(
            model=model,
            name=name,
            busi_train_loader=busi_train_loader,
            busi_val_loader=busi_val_loader,
            breakhis_loader=breakhis_loader,
            class_weights=class_weights,
            epochs=CFG.msvte_epochs,
            lr=CFG.msvte_lr,
            weight_decay=CFG.msvte_weight_decay,
            lambda_domain=MSVTE_LAMBDA_DOMAIN,
            domain_loss_weight=MSVTE_DOMAIN_LOSS_WT,
            grad_clip=GRAD_CLIP
        )

    labels, preds, uncert = cd_msvte_mc_ensemble_predict(models, busi_test_loader, mc_samples=MSVTE_MC_SAMPLES)

    print("\n[CD-MSVTE] BUSI Test Classification Report:")
    print(classification_report(labels, preds, target_names=BUSI_CLASSES, digits=4))
    cm = confusion_matrix(labels, preds)
    print("[CD-MSVTE] Confusion Matrix:\n", cm)

    out_path = RESULTS_DIR / "cd_msvte_u_test_report.txt"
    with open(out_path, "w", encoding="utf-8") as f:
        f.write("Cross-Domain MSVTE-U (Option A): Domain-Adaptive Multi-Scale ViT Ensemble with Uncertainty\n\n")
        f.write(classification_report(labels, preds, target_names=BUSI_CLASSES, digits=4))
        f.write("\n\nConfusion Matrix:\n")
        f.write(str(cm))
        f.write("\n\nUncertainty (entropy) stats:\n")
        f.write(f"mean={float(np.mean(uncert)):.6f}, std={float(np.std(uncert)):.6f}, "
                f"min={float(np.min(uncert)):.6f}, max={float(np.max(uncert)):.6f}\n")

    print(f"[CD-MSVTE] Saved report to: {out_path}")
    return models, names


# BUSI-only MSVTE-U -- the baseline arm of the paper's ablation (HViTE-U)
def train_msvte_u_bus_only():
    """Original BUSI-only MSVTE-U (kept for reference)."""
    return train_msvte_u()


# ---------------------------
# BUSI-only MSVTE-U, no domain-adversarial term. This is the baseline arm.
# ---------------------------
def build_msvte_models():
    names = [
        "vit_small_patch16_224.augreg_in21k",
        "vit_base_patch16_224.augreg_in21k",
        "deit_small_patch16_224"
    ]
    models = [SingleViTClassifier(n, num_classes=len(BUSI_CLASSES), dropout_p=MSVTE_DROPOUT).to(device)
              for n in names]
    return models, names

def train_msvte_u_bus_only():
    print("\n=== Approach 2 (ORIGINAL): MSVTE-U Training (BUSI-only) ===")
    train_tf = get_basic_tf(is_train=True)
    test_tf  = get_basic_tf(is_train=False)

    busi_train_ds = BUSIDataset(busi_train_s, transform=train_tf)
    busi_val_ds   = BUSIDataset(busi_val_s, transform=test_tf)
    busi_test_ds  = BUSIDataset(busi_test_s, transform=test_tf)

    class_weights = compute_busi_class_weights(busi_train_s)
    sample_weights = [1.0 / (class_weights[int(lbl)].item() + 1e-8) for _, lbl in busi_train_s]
    sampler = torch.utils.data.WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

    train_loader = DataLoader(busi_train_ds, batch_size=CFG.msvte_batch_size, sampler=sampler,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(busi_val_ds, batch_size=CFG.msvte_batch_size, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    test_loader  = DataLoader(busi_test_ds, batch_size=CFG.msvte_batch_size, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)

    models, names = build_msvte_models()
    for model, name in zip(models, names):
        best_path = train_single_vit(model, train_loader, val_loader, class_weights,
                                     name=name, epochs=CFG.msvte_epochs, lr=CFG.msvte_lr, weight_decay=CFG.msvte_weight_decay)
        model.load_state_dict(torch.load(best_path, map_location=device))

    labels, preds, uncert = cd_msvte_mc_ensemble_predict(models, test_loader, mc_samples=MSVTE_MC_SAMPLES)

    print("\n[MSVTE-U BUSI-only] Test report:")
    print(classification_report(labels, preds, target_names=BUSI_CLASSES, digits=4))
    return models, names


In [20]:
class PathDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        return img, label


class SimCLRDataset(Dataset):
    def __init__(self, base_dataset, transform):
        self.base = base_dataset
        self.transform = transform

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, _ = self.base[idx]   # label ignored (self-supervised)
        v1 = self.transform(img)
        v2 = self.transform(img)
        return v1, v2



def nt_xent_loss(z1, z2, temperature=0.5):
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)
    B = z1.size(0)
    z = torch.cat([z1, z2], dim=0)  # 2B x D
    sim = torch.matmul(z, z.T) / temperature

    mask = torch.eye(2*B, device=z.device).bool()
    sim = sim.masked_fill(mask, -9e15)

    positives = torch.cat([torch.diag(sim, B), torch.diag(sim, -B)], dim=0)
    denom = torch.logsumexp(sim, dim=1)

    loss = -positives + denom
    return loss.mean()


In [12]:
class ViTEncoder(nn.Module):
    def __init__(self, backbone_name="vit_small_patch16_224.augreg_in21k"):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0)
        self.embed_dim = self.backbone.num_features

    def forward(self, x):
        return self.backbone(x)


class ProjectionHead(nn.Module):
    def __init__(self, in_dim, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, in_dim),
            nn.ReLU(inplace=True),
            nn.Linear(in_dim, out_dim),
        )

    def forward(self, x):
        return self.net(x)


In [21]:
def train_simclr_encoder():
    print("\n=== SimCLR pretraining on BUSI (all splits) ===")
    simclr_tf = get_simclr_tf()

    all_samples = busi_train_s + busi_val_s + busi_test_s
    base_ds = PathDataset(all_samples)
    simclr_ds = SimCLRDataset(base_ds, simclr_tf)

    loader = DataLoader(simclr_ds, batch_size=CFG.simclr_batch_size,
                        shuffle=True, num_workers=0, drop_last=True)

    encoder = ViTEncoder().to(device)
    proj_head = ProjectionHead(encoder.embed_dim).to(device)

    optimizer = torch.optim.AdamW(
        list(encoder.parameters()) + list(proj_head.parameters()),
        lr=CFG.simclr_lr,
        weight_decay=CFG.simclr_weight_decay,
    )

    losses = []
    for epoch in range(1, CFG.simclr_epochs + 1):
        encoder.train()
        proj_head.train()
        running = 0.0
        pbar = tqdm(loader, desc=f"[SimCLR] Epoch {epoch}/{CFG.simclr_epochs}")
        for v1, v2, _ in pbar:
            v1 = v1.to(device)
            v2 = v2.to(device)
            h1 = encoder(v1)
            h2 = encoder(v2)
            z1 = proj_head(h1)
            z2 = proj_head(h2)
            loss = nt_xent_loss(z1, z2, temperature=CFG.simclr_temperature)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.3f}"})

        avg_loss = running / max(1, len(loader))
        losses.append(avg_loss)
        print(f"[SimCLR] Epoch {epoch} - Loss: {avg_loss:.4f}")

    enc_path = CHECKPOINT_DIR / "hcml_simclr_encoder.pth"
    torch.save(encoder.state_dict(), enc_path)
    print("Saved SimCLR encoder to:", enc_path)

    epochs_axis = range(1, len(losses) + 1)
    plt.figure()
    plt.plot(epochs_axis, losses)
    plt.xlabel("Epoch"); plt.ylabel("Loss")
    plt.title("SimCLR Pretraining Loss")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "hcml_simclr_loss.png")
    plt.close()

    return encoder


In [14]:
class EpisodicSampler:
    def __init__(self, samples, n_way, k_shot, q_queries):
        self.samples = samples
        self.n_way = n_way
        self.k_shot = k_shot
        self.q_queries = q_queries

        self.class_to_indices = defaultdict(list)
        for idx, (_, y) in enumerate(samples):
            self.class_to_indices[y].append(idx)
        self.classes = list(self.class_to_indices.keys())

    def sample_episode(self):
        selected = random.sample(self.classes, self.n_way)
        support_x, support_y = [], []
        query_x, query_y = [], []

        for new_label, cls in enumerate(selected):
            idxs = self.class_to_indices[cls]
            if len(idxs) < self.k_shot + self.q_queries:
                chosen = random.choices(idxs, k=self.k_shot + self.q_queries)
            else:
                chosen = random.sample(idxs, self.k_shot + self.q_queries)
            s_idx = chosen[:self.k_shot]
            q_idx = chosen[self.k_shot:]

            for i in s_idx:
                support_x.append(self.samples[i][0])
                support_y.append(new_label)
            for i in q_idx:
                query_x.append(self.samples[i][0])
                query_y.append(new_label)

        return support_x, support_y, query_x, query_y


class BUSIPathDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        img = self.transform(img)
        return img, self.labels[idx]


In [15]:
class ProtoNetHead(nn.Module):
    def __init__(self, in_dim, temperature=1.0):
        super().__init__()
        self.scale = nn.Parameter(torch.tensor(temperature, dtype=torch.float32))
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, in_dim),
            nn.ReLU(inplace=True),
            nn.Linear(in_dim, in_dim),
        )

    def forward(self, support, query, support_labels, n_way):
        support = self.mlp(support)
        query = self.mlp(query)

        support = F.normalize(support, dim=1)
        query = F.normalize(query, dim=1)

        prototypes = []
        for c in range(n_way):
            idxs = (support_labels == c)
            prototypes.append(support[idxs].mean(dim=0))
        prototypes = torch.stack(prototypes, dim=0)

        logits = self.scale * torch.matmul(query, prototypes.t())
        return logits


In [22]:
# ============================================================
# Approach 3: Cross-Domain HCML (CD-HCML)
# SimCLR on BreakHis + BUSI  +  ProtoNet episodes (support=BreakHis, query=BUSI)
# ============================================================

# ---- Safe defaults if CFG doesn't define these ----
NUM_WORKERS = getattr(CFG, "num_workers", 0)
GRAD_CLIP   = getattr(CFG, "grad_clip", 1.0)

def _filter_busi_benign_malignant(samples):
    """Keep only BUSI benign/malignant for true cross-domain alignment with BreakHis (2-way)."""
    keep = []
    for p, y in samples:
        if y in (BUSI_CLASS_TO_IDX["benign"], BUSI_CLASS_TO_IDX["malignant"]):
            keep.append((p, y))
    return keep

def _map_busi_to_binary(y):
    """Map BUSI benign->0, malignant->1. (Normal is not used in CD-HCML episodes.)"""
    if y == BUSI_CLASS_TO_IDX["benign"]:
        return 0
    if y == BUSI_CLASS_TO_IDX["malignant"]:
        return 1
    raise ValueError("CD-HCML uses only benign/malignant BUSI samples.")


def train_simclr_encoder_cross_domain():
    """
    UPDATED SimCLR:
    - Build an unlabeled pool from BreakHis + BUSI (all splits)
    - Train ViT encoder + projection head using NT-Xent
    """
    print("\n=== SimCLR pretraining (UPDATED): BreakHis + BUSI (mixed, unlabeled) ===")
    simclr_tf = get_simclr_tf()

    # Pooled BreaKHis + BUSI images for SimCLR; labels are not used here
    busi_all = busi_train_s + busi_val_s + busi_test_s
    breakhis_ds = BreakHisDataset(CFG.BREAKHIS_ROOT, transform=None)

    mixed_samples = []
    mixed_samples.extend([(p, 0) for (p, _) in busi_all])
    if hasattr(breakhis_ds, "samples"):
        mixed_samples.extend([(p, 0) for (p, _) in breakhis_ds.samples])

    base_ds = PathDataset(mixed_samples)
    simclr_ds = SimCLRDataset(base_ds, transform=simclr_tf)
    loader = DataLoader(simclr_ds, batch_size=CFG.simclr_batch_size, shuffle=True,
                        num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)

    encoder = ViTEncoder().to(device)  # uses default ViT backbone from your notebook
    proj = ProjectionHead(encoder.embed_dim).to(device)

    optimizer = torch.optim.AdamW(list(encoder.parameters()) + list(proj.parameters()),
                                  lr=CFG.simclr_lr, weight_decay=CFG.simclr_weight_decay)

    losses = []
    for epoch in range(1, CFG.simclr_epochs + 1):
        encoder.train()
        proj.train()
        epoch_loss = 0.0
        n_steps = 0

        for x1, x2 in loader:
            x1 = x1.to(device)
            x2 = x2.to(device)

            h1 = encoder(x1)
            h2 = encoder(x2)
            z1 = proj(h1)
            z2 = proj(h2)

            loss = nt_xent_loss(z1, z2, temperature=CFG.simclr_temperature)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            if GRAD_CLIP is not None and GRAD_CLIP > 0:
                torch.nn.utils.clip_grad_norm_(list(encoder.parameters()) + list(proj.parameters()), GRAD_CLIP)
            optimizer.step()

            epoch_loss += loss.item()
            n_steps += 1

        avg = epoch_loss / max(n_steps, 1)
        losses.append(avg)
        print(f"[CD-SimCLR] Epoch {epoch:02d}/{CFG.simclr_epochs} | Loss {avg:.4f}")

    enc_path = CHECKPOINT_DIR / "cd_hcml_simclr_encoder.pth"
    torch.save(encoder.state_dict(), enc_path)
    print(f"[CD-SimCLR] Saved encoder to: {enc_path}")

    try:
        plt.figure()
        plt.plot(losses)
        plt.title("CD-HCML SimCLR Pretraining Loss (BreakHis + BUSI)")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.tight_layout()
        plt.savefig(RESULTS_DIR / "cd_hcml_simclr_loss.png", dpi=160)
        plt.close()
    except Exception as e:
        print("Plotting failed:", e)

    return encoder


def train_cd_hcml_protonet():
    """
    UPDATED Approach 3:
    - SimCLR encoder pretrained on BreakHis + BUSI
    - Cross-domain ProtoNet episodes:
        Support = BreakHis (2-way)
        Query   = BUSI (2-way, benign/malignant only)
    NOTE: BUSI 'normal' is excluded here because BreakHis has only benign/malignant.
    """
    print("\n=== Approach 3 (UPDATED): CD-HCML Training (BreakHis -> BUSI) ===")

    # (1) Load / train SimCLR encoder (cross-domain)
    enc_path = CHECKPOINT_DIR / "cd_hcml_simclr_encoder.pth"
    encoder = ViTEncoder().to(device)
    if enc_path.exists():
        encoder.load_state_dict(torch.load(enc_path, map_location=device))
        print(f"[CD-HCML] Loaded SimCLR encoder: {enc_path}")
    else:
        encoder = train_simclr_encoder_cross_domain()

    # Encoder is frozen: only the ProtoNet head is trained
    freeze_encoder = True
    if freeze_encoder:
        for p in encoder.parameters():
            p.requires_grad = False
        encoder.eval()
        print("[CD-HCML] Encoder frozen for ProtoNet training.")

    # (2) Build episode pools
    breakhis_ds = BreakHisDataset(CFG.BREAKHIS_ROOT, transform=None)
    breakhis_samples = breakhis_ds.samples if hasattr(breakhis_ds, "samples") else []

    busi_train_bin = [(p, _map_busi_to_binary(y)) for (p, y) in _filter_busi_benign_malignant(busi_train_s)]
    busi_val_bin   = [(p, _map_busi_to_binary(y)) for (p, y) in _filter_busi_benign_malignant(busi_val_s)]
    busi_test_bin  = [(p, _map_busi_to_binary(y)) for (p, y) in _filter_busi_benign_malignant(busi_test_s)]

    from collections import defaultdict
    bh_by_class = defaultdict(list)
    for p, y in breakhis_samples:
        if int(y) in (0, 1):
            bh_by_class[int(y)].append(p)

    def build_by_class(samples):
        d = defaultdict(list)
        for p, y in samples:
            d[int(y)].append(p)
        return d

    busi_train_by = build_by_class(busi_train_bin)
    busi_val_by   = build_by_class(busi_val_bin)
    busi_test_by  = build_by_class(busi_test_bin)

    N_WAY = 2
    K_SHOT = CFG.meta_k_shot
    Q_QUERY = CFG.meta_q_queries

    support_tf = get_basic_tf(is_train=True)
    query_tf   = get_basic_tf(is_train=False)

    def sample_cd_episode(bh_dict, busi_dict, k_shot, q_query):
        classes = [0, 1]
        s_paths, s_y, q_paths, q_y = [], [], [], []
        for cls in classes:
            sp = random.sample(bh_dict[cls], k=min(k_shot, len(bh_dict[cls])))
            qp = random.sample(busi_dict[cls], k=min(q_query, len(busi_dict[cls])))
            s_paths.extend(sp); s_y.extend([cls]*len(sp))
            q_paths.extend(qp); q_y.extend([cls]*len(qp))
        return s_paths, s_y, q_paths, q_y

    # (3) ProtoNet head (trainable)
    head = ProtoNetHead(encoder.embed_dim, temperature=1.0).to(device)
    optimizer = torch.optim.AdamW(head.parameters(), lr=CFG.meta_lr_head, weight_decay=CFG.meta_weight_decay)
    criterion = nn.CrossEntropyLoss()

    # (4) Meta-training (iterations)
    print(f"[CD-HCML] Episodic training: {N_WAY}-way, {K_SHOT}-shot, {Q_QUERY} queries/class")
    best_val = -1.0
    best_path = CHECKPOINT_DIR / "cd_hcml_protonet_head.pth"

    def eval_cd(busi_dict, n_episodes=50):
        encoder.eval()
        head.eval()
        accs = []
        for _ in range(n_episodes):
            s_paths, s_y, q_paths, q_y = sample_cd_episode(bh_by_class, busi_dict, K_SHOT, Q_QUERY)

            s_ds = BUSIPathDataset(s_paths, s_y, transform=support_tf)
            q_ds = BUSIPathDataset(q_paths, q_y, transform=query_tf)

            s_x, s_lbl = next(iter(DataLoader(s_ds, batch_size=len(s_ds), shuffle=False)))
            q_x, q_lbl = next(iter(DataLoader(q_ds, batch_size=len(q_ds), shuffle=False)))

            s_x = s_x.to(device); q_x = q_x.to(device)
            s_lbl = torch.tensor(s_lbl, dtype=torch.long, device=device)
            q_lbl = torch.tensor(q_lbl, dtype=torch.long, device=device)

            with torch.no_grad():
                s_feat = encoder(s_x)
                q_feat = encoder(q_x)
                logits = head(s_feat, q_feat, s_lbl, n_way=N_WAY)
                preds = logits.argmax(dim=1)
                accs.append((preds == q_lbl).float().mean().item())
        return float(np.mean(accs)), float(np.std(accs))

    for it in range(1, CFG.meta_iters + 1):
        head.train()

        s_paths, s_y, q_paths, q_y = sample_cd_episode(bh_by_class, busi_train_by, K_SHOT, Q_QUERY)

        s_ds = BUSIPathDataset(s_paths, s_y, transform=support_tf)
        q_ds = BUSIPathDataset(q_paths, q_y, transform=support_tf)

        s_x, s_lbl = next(iter(DataLoader(s_ds, batch_size=len(s_ds), shuffle=False)))
        q_x, q_lbl = next(iter(DataLoader(q_ds, batch_size=len(q_ds), shuffle=False)))

        s_x = s_x.to(device); q_x = q_x.to(device)
        s_lbl = torch.tensor(s_lbl, dtype=torch.long, device=device)
        q_lbl = torch.tensor(q_lbl, dtype=torch.long, device=device)

        with torch.no_grad():
            s_feat = encoder(s_x)
            q_feat = encoder(q_x)

        logits = head(s_feat, q_feat, s_lbl, n_way=N_WAY)
        loss = criterion(logits, q_lbl)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        if GRAD_CLIP is not None and GRAD_CLIP > 0:
            torch.nn.utils.clip_grad_norm_(head.parameters(), GRAD_CLIP)
        optimizer.step()

        if it % 50 == 0:
            val_mean, val_std = eval_cd(busi_val_by, n_episodes=CFG.meta_batch_episodes*5)
            print(f"[CD-HCML] Iter {it:04d}/{CFG.meta_iters} | Loss {loss.item():.4f} | Val ep-acc {val_mean:.4f} ± {val_std:.4f}")
            if val_mean > best_val:
                best_val = val_mean
                torch.save(head.state_dict(), best_path)

    # Load best head
    if best_path.exists():
        head.load_state_dict(torch.load(best_path, map_location=device))
        print(f"[CD-HCML] Loaded best ProtoNet head: {best_path}")

    test_mean, test_std = eval_cd(busi_test_by, n_episodes=CFG.meta_batch_episodes*5)
    print(f"\n[CD-HCML] TEST ({N_WAY}-way {K_SHOT}-shot): {test_mean:.4f} ± {test_std:.4f}")

    out_path = RESULTS_DIR / "cd_hcml_fewshot_results.txt"
    with open(out_path, "w", encoding="utf-8") as f:
        f.write("CD-HCML (BreakHis -> BUSI) Few-Shot Results\n")
        f.write(f"N-way={N_WAY}, K-shot={K_SHOT}, Q-query={Q_QUERY}\n")
        f.write(f"Best Val Episode Acc: {best_val:.6f}\n")
        f.write(f"Test Episode Acc (mean±std): {test_mean:.6f} ± {test_std:.6f}\n")

    print(f"[CD-HCML] Saved few-shot results to: {out_path}")
    return encoder, head


# BUSI-only HCML, kept for reference
def train_hcml_protonet_bus_only():
    """Original BUSI-only HCML (kept for reference)."""
    return train_hcml_protonet()

In [17]:
# ssdavt_model = train_ssdavt()


In [18]:
msvte_models, msvte_names = train_cd_msvte_u()



=== Approach 2 (UPDATED): Cross-Domain MSVTE-U Training ===
[BreakHis] Scanning benign folder:    D:\My Thesis\bushra\data\breakhis\BreaKHis_v1\histology_slides\breast\benign\SOB
[BreakHis] Scanning malignant folder: D:\My Thesis\bushra\data\breakhis\BreaKHis_v1\histology_slides\breast\malignant\SOB
[BreakHis] Loaded 7909 images from 8 classes.
[BreakHis] Classes: ['adenosis', 'fibroadenoma', 'phyllodes_tumor', 'tubular_adenoma', 'ductal_carcinoma', 'lobular_carcinoma', 'mucinous_carcinoma', 'papillary_carcinoma']
BUSI train class counts: Counter({0: 272, 1: 127, 2: 79})
Class weights (inverse freq): tensor([1.7574, 3.7638, 6.0506], device='cuda:0')
[CD-MSVTE] vit_small_patch16_224.augreg_in21k | Epoch 01/30 | Loss 2.5368 (cls 1.5902, dom 0.9465, λ 0.00) | Val Acc 0.5687
[CD-MSVTE] vit_small_patch16_224.augreg_in21k | Epoch 02/30 | Loss 2.1055 (cls 1.2659, dom 0.8395, λ 0.01) | Val Acc 0.5687
[CD-MSVTE] vit_small_patch16_224.augreg_in21k | Epoch 03/30 | Loss 2.0006 (cls 1.1304, dom 0.

C:\Users\FAST\AppData\Local\Temp\ipykernel_15600\1373206803.py:160: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_path, map_location=de

[CD-MSVTE] vit_base_patch16_224.augreg_in21k | Epoch 01/30 | Loss 3.4173 (cls 2.3711, dom 1.0462, λ 0.00) | Val Acc 0.1688
[CD-MSVTE] vit_base_patch16_224.augreg_in21k | Epoch 02/30 | Loss 2.3536 (cls 1.4817, dom 0.8719, λ 0.01) | Val Acc 0.5687
[CD-MSVTE] vit_base_patch16_224.augreg_in21k | Epoch 03/30 | Loss 2.2264 (cls 1.3309, dom 0.8956, λ 0.02) | Val Acc 0.5687
[CD-MSVTE] vit_base_patch16_224.augreg_in21k | Epoch 04/30 | Loss 2.3953 (cls 1.3439, dom 1.0514, λ 0.03) | Val Acc 0.5687
[CD-MSVTE] vit_base_patch16_224.augreg_in21k | Epoch 05/30 | Loss 2.9093 (cls 1.2530, dom 1.6563, λ 0.04) | Val Acc 0.5500
[CD-MSVTE] vit_base_patch16_224.augreg_in21k | Epoch 06/30 | Loss 14.7261 (cls 1.0441, dom 13.6819, λ 0.05) | Val Acc 0.5687
[CD-MSVTE] vit_base_patch16_224.augreg_in21k | Epoch 07/30 | Loss 5.4406 (cls 1.5785, dom 3.8621, λ 0.06) | Val Acc 0.5625
[CD-MSVTE] vit_base_patch16_224.augreg_in21k | Epoch 08/30 | Loss 2.0573 (cls 1.2301, dom 0.8272, λ 0.07) | Val Acc 0.2625
[CD-MSVTE] vit

In [23]:
hcml_encoder, hcml_head = train_cd_hcml_protonet()



=== Approach 3 (UPDATED): CD-HCML Training (BreakHis -> BUSI) ===

=== SimCLR pretraining (UPDATED): BreakHis + BUSI (mixed, unlabeled) ===
[BreakHis] Scanning benign folder:    D:\My Thesis\bushra\data\breakhis\BreaKHis_v1\histology_slides\breast\benign\SOB
[BreakHis] Scanning malignant folder: D:\My Thesis\bushra\data\breakhis\BreaKHis_v1\histology_slides\breast\malignant\SOB
[BreakHis] Loaded 7909 images from 8 classes.
[BreakHis] Classes: ['adenosis', 'fibroadenoma', 'phyllodes_tumor', 'tubular_adenoma', 'ductal_carcinoma', 'lobular_carcinoma', 'mucinous_carcinoma', 'papillary_carcinoma']
[CD-SimCLR] Epoch 01/20 | Loss 3.4494
[CD-SimCLR] Epoch 02/20 | Loss 3.2452
[CD-SimCLR] Epoch 03/20 | Loss 3.2110
[CD-SimCLR] Epoch 04/20 | Loss 3.1884
[CD-SimCLR] Epoch 05/20 | Loss 3.1755
[CD-SimCLR] Epoch 06/20 | Loss 3.1556
[CD-SimCLR] Epoch 07/20 | Loss 3.1503
[CD-SimCLR] Epoch 08/20 | Loss 3.1359
[CD-SimCLR] Epoch 09/20 | Loss 3.1283
[CD-SimCLR] Epoch 10/20 | Loss 3.1226
[CD-SimCLR] Epoch 1

C:\Users\FAST\AppData\Local\Temp\ipykernel_15600\1385851428.py:222: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  s_lbl = torch.tensor(s_lbl, dtype=torch.long, device=device)
C:\Users\FAST\AppData\Local\Temp\ipykernel_15600\1385851428.py:223: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  q_lbl = torch.tensor(q_lbl, dtype=torch.long, device=device)
C:\Users\FAST\AppData\Local\Temp\ipykernel_15600\1385851428.py:199: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  s_lbl = torch.tensor(s_lbl, dtype=torch.long, device=device)
C:\Users\FAST\AppDa

[CD-HCML] Iter 0050/400 | Loss 0.6004 | Val ep-acc 0.7600 ± 0.0718
[CD-HCML] Iter 0100/400 | Loss 0.4497 | Val ep-acc 0.7688 ± 0.0842
[CD-HCML] Iter 0150/400 | Loss 0.3983 | Val ep-acc 0.7775 ± 0.1270
[CD-HCML] Iter 0200/400 | Loss 0.4169 | Val ep-acc 0.8488 ± 0.0711
[CD-HCML] Iter 0250/400 | Loss 0.4744 | Val ep-acc 0.7825 ± 0.0810
[CD-HCML] Iter 0300/400 | Loss 0.4114 | Val ep-acc 0.7375 ± 0.0650
[CD-HCML] Iter 0350/400 | Loss 0.4376 | Val ep-acc 0.8163 ± 0.0711
[CD-HCML] Iter 0400/400 | Loss 0.4730 | Val ep-acc 0.7863 ± 0.1265
[CD-HCML] Loaded best ProtoNet head: D:\My Thesis\hybrid-three-approach\models\cd_hcml_protonet_head.pth


C:\Users\FAST\AppData\Local\Temp\ipykernel_15600\1385851428.py:247: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  head.load_state_dict(torch.load(best_path, map_location=dev


[CD-HCML] TEST (2-way 5-shot): 0.7825 ± 0.0841
[CD-HCML] Saved few-shot results to: D:\My Thesis\hybrid-three-approach\results\cd_hcml_fewshot_results.txt
